# Pass protection: Pocket Integrity Score (PIS) & Blitz Time Pressure (BTP)

NFL Big Data Bowl 2023 data (2021 season, weeks 1-8). Two linked tracking metrics:

* **PIS (0-100)**: how clean the QB's pocket stays from snap to throw. `Z = 0.45·z(cushion) + 0.40·z(clean_time_pct) + 0.15·z(pocket_area)`, `PIS = 100·Φ(Z / sd(Z))` → 50 = league average, 84 ≈ one standard deviation cleaner.
  * `cushion` = mean distance QB ↔ nearest rusher over the frames (winsorised 1-99%)
  * `clean_time_pct` = share of frames with no rusher within 2 yd
  * `pocket_area` = mean convex-hull area of the pass blockers (secondary: a large area can be a stretched line)
  * `PIS_adj` = 50 + residual of an OLS of PIS on time to throw (quick throws face less pressure)
* **BTP / time to pressure**: first time after the snap a rusher enters a 2-yd circle around the QB. Plays where the ball comes out first are **right-censored** → Kaplan-Meier curves (implemented with numpy).

All code lives in `src/`; this notebook runs top to bottom and re-creates every figure in `outputs/`.

In [1]:
import warnings
import pandas as pd
import plotly.io as pio

from src import config as C
from src import metrics as M
from src import viz
from src.data_loading import describe_raw
from src.pipeline import analyse, compute, pick_examples

pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.width", 200, "display.max_columns", 30)
warnings.filterwarnings("ignore", category=RuntimeWarning)

## 1. Inspect the raw data

In [2]:
describe_raw()

===== games.csv  shape=(122, 7)
['gameId', 'season', 'week', 'gameDate', 'gameTimeEastern', 'homeTeamAbbr', 'visitorTeamAbbr']
       gameId  season  week    gameDate gameTimeEastern homeTeamAbbr visitorTeamAbbr
0  2021090900    2021     1  09/09/2021        20:20:00           TB             DAL
1  2021091200    2021     1  09/12/2021        13:00:00          ATL             PHI
2  2021091201    2021     1  09/12/2021        13:00:00          BUF             PIT 

===== plays.csv  shape=(8557, 32)
['gameId', 'playId', 'playDescription', 'quarter', 'down', 'yardsToGo', 'possessionTeam', 'defensiveTeam', 'yardlineSide', 'yardlineNumber', 'gameClock', 'preSnapHomeScore', 'preSnapVisitorScore', 'passResult', 'penaltyYards', 'prePenaltyPlayResult', 'playResult', 'foulName1', 'foulNFLId1', 'foulName2', 'foulNFLId2', 'foulName3', 'foulNFLId3', 'absoluteYardlineNumber', 'offenseFormation', 'personnelO', 'defendersInBox', 'personnelD', 'dropBackType', 'pff_playAction', 'pff_passCoverage', 'pff_

===== pffScoutingData.csv  shape=(188254, 15)
['gameId', 'playId', 'nflId', 'pff_role', 'pff_positionLinedUp', 'pff_hit', 'pff_hurry', 'pff_sack', 'pff_beatenByDefender', 'pff_hitAllowed', 'pff_hurryAllowed', 'pff_sackAllowed', 'pff_nflIdBlockedPlayer', 'pff_blockType', 'pff_backFieldBlock']
       gameId  playId  nflId    pff_role pff_positionLinedUp  pff_hit  pff_hurry  pff_sack  pff_beatenByDefender  pff_hitAllowed  pff_hurryAllowed  pff_sackAllowed  pff_nflIdBlockedPlayer pff_blockType  pff_backFieldBlock
0  2021090900      97  25511        Pass                  QB      NaN        NaN       NaN                   NaN             NaN               NaN              NaN                     NaN           NaN                 NaN
1  2021090900      97  35481  Pass Route                TE-L      NaN        NaN       NaN                   NaN             NaN               NaN              NaN                     NaN           NaN                 NaN
2  2021090900      97  35634  Pass Route 

## 2. Preprocess: development run on weeks 1-2

Filters: `passResult ∈ {C, I, S, IN}` (scrambles R dropped), ≥1 `Pass Rush` player, a `ball_snap` event. Window = `ball_snap` → first of `pass_forward / pass_shovel / qb_sack / qb_strip_sack` (fallback `autoevent_passforward`). Coordinates flipped so the offense always moves toward +x. Frames cached as parquet in `data/processed/`.

In [3]:
dev = compute(weeks=[1, 2])
dev["plays"][["cushion", "clean_time_pct", "pocket_area", "time_to_throw", "PIS", "time_to_pressure", "pressure_event"]].describe().round(2)

                                      step  plays_remaining
                    all plays in plays.csv             8557
passResult in C/I/S/IN (drops scrambles R)             8108
           at least one 'Pass Rush' player             8107
                  has a 'Pass' (QB) player             8107


week 1: 1118 plays with snap, 1116 with metrics (2s)


week 2: 998 plays with snap, 996 with metrics (4s)
                                      step  plays_remaining
                    all plays in plays.csv             8557
passResult in C/I/S/IN (drops scrambles R)             8108
           at least one 'Pass Rush' player             8107
                  has a 'Pass' (QB) player             8107
                                 weeks 1-2             2123
     has ball_snap + end event in tracking             2112
 frames_rows  rows_strictly_outside_field  plays_without_snap  plays_without_end_event  plays_using_autoevent_end  ttt_outliers  week
      782391                           92                   3                        2                          6             2     1
      720636                           47                   4                        2                          3             0     2
time_to_throw outliers (outside 0.5-8.0s, flagged not dropped): 2 · plays lost in tracking: 11


,cushion,clean_time_pct,pocket_area,time_to_throw,PIS,time_to_pressure,pressure_event
count,2112.00,2112.00,2112.00,2112.00,2112.00,2112.00,2112.00
mean,4.36,0.94,8.91,2.99,53.38,2.71,0.34
std,0.75,0.11,5.32,1.04,26.30,0.79,0.47
min,1.61,0.37,1.16,0.90,0.00,0.10,0.00
25%,3.95,0.93,5.47,2.30,35.28,2.20,0.00
50%,4.45,1.00,7.58,2.80,60.24,2.60,0.00
75%,4.87,1.00,10.83,3.40,73.73,3.10,1.00
max,8.21,1.00,49.38,9.80,99.88,7.90,1.00


## 3. Full run: weeks 1-8

In [4]:
res = compute(weeks=range(1, 9))
df = res["plays"]
print(f"{len(df)} dropbacks kept · PIS slope vs time to throw: {df.attrs['ttt_slope']:.1f} points per second")
out = analyse(res, make_figures=True, verbose=False)

                                      step  plays_remaining
                    all plays in plays.csv             8557
passResult in C/I/S/IN (drops scrambles R)             8108
           at least one 'Pass Rush' player             8107
                  has a 'Pass' (QB) player             8107


week 1: 1118 plays with snap, 1116 with metrics (2s)


week 2: 998 plays with snap, 996 with metrics (4s)


week 3: 1069 plays with snap, 1064 with metrics (6s)


week 4: 1051 plays with snap, 1041 with metrics (8s)


week 5: 1049 plays with snap, 1046 with metrics (9s)


week 6: 952 plays with snap, 950 with metrics (11s)


week 7: 860 plays with snap, 859 with metrics (12s)


week 8: 987 plays with snap, 985 with metrics (14s)
                                      step  plays_remaining
                    all plays in plays.csv             8557
passResult in C/I/S/IN (drops scrambles R)             8108
           at least one 'Pass Rush' player             8107
                  has a 'Pass' (QB) player             8107
                                 weeks 1-8             8107
     has ball_snap + end event in tracking             8057
 frames_rows  rows_strictly_outside_field  plays_without_snap  plays_without_end_event  plays_using_autoevent_end  ttt_outliers  week
      782391                           92                   3                        2                          6             2     1
      720636                           47                   4                        2                          3             0     2
      776365                           53                   2                        5                          6             

## 4. Metric 1: Pocket Integrity Score
PIS must always be read against time to throw.

In [5]:
out["figures"]["pis_vs_time_to_throw"].show()

Weight sensitivity: team ranking stability and play-level AUC under alternative weights.

In [6]:
out["weight_sensitivity"]

,weights,AUC_pressure,spearman_team_rank_vs_baseline,spearman_team_PISadj_vs_pressure_rate,top5_teams
0,baseline (.45/.40/.15),0.798,1.000,-0.133,"ARI, BAL, KC, LA, BUF"
1,equal (1/3 each),0.744,0.967,-0.008,"ARI, BAL, BUF, KC, LA"
2,no area (.5/.5/0),0.816,0.993,-0.135,"ARI, BAL, LA, KC, BUF"
3,cushion heavy (.7/.2/.1),0.780,0.988,-0.109,"BAL, ARI, KC, LA, BUF"
4,clean heavy (.2/.7/.1),0.826,0.976,-0.176,"ARI, LA, KC, BAL, BUF"
5,area heavy (.3/.3/.4),0.716,0.961,-0.014,"ARI, BAL, BUF, KC, LA"


Results with and without play-action and pocket-roll (PR) plays:

In [7]:
out["subsets"]

,subset,n_plays,mean_PIS,mean_time_to_throw_s,pff_pressure_rate,AUC_PIS_vs_pff_pressure,KM_median_time_to_pressure_s
0,all kept plays,8057,53.385,3.036,0.363,0.798,3.5
1,excluding play-action,6162,54.488,2.909,0.358,0.806,3.2
2,excluding pocket-roll (PR),7994,53.205,3.037,0.364,0.798,3.5
3,excluding play-action and PR,6104,54.275,2.909,0.359,0.807,3.2
4,play-action only,1895,49.800,3.450,0.379,0.772,4.3


## 5. Metric 2: time to pressure (Kaplan-Meier)
The PFF rusher count is the blitz variable; defenders in the box is only a secondary pre-snap indicator.

In [8]:
out["figures"]["km_rushers"].show()
print(f"Log-rank test, 4 vs 6+ rushers: p = {out['logrank_4_vs_6']:.2g}")

Log-rank test, 4 vs 6+ rushers: p = 4.8e-16


In [9]:
out["figures"]["km_box_small_multiples"].show()
out["figures"]["km_coverage_formation"].show()

In [10]:
out["km_tables"]

,variable,group,n_plays,pressure_events,censored_pct,median_time_to_pressure_s,S(1.5s),S(2.5s),mean_time_to_throw_s
0,rush_bin,≤3,410,86,79.0,5.4,1.000,0.925,3.28
1,rush_bin,4,5620,1831,67.4,3.6,0.990,0.856,3.08
2,rush_bin,5,1618,609,62.4,3.1,0.987,0.775,2.89
3,rush_bin,6+,409,155,62.1,3.0,0.980,0.678,2.70
4,box_bin,≤5,2050,752,63.3,3.2,0.998,0.823,3.03
5,box_bin,6,3770,1230,67.4,3.5,0.995,0.837,3.01
6,box_bin,7,1727,549,68.2,3.8,0.977,0.839,3.06
7,box_bin,8+,504,148,70.6,4.3,0.954,0.868,3.17
8,pff_passCoverageType,Man,2346,786,66.5,3.4,0.991,0.813,2.95
9,pff_passCoverageType,Zone,5263,1740,66.9,3.6,0.989,0.846,3.05


Sensitivity of the pressure radius (1.5 / 2 / 3 yd):

In [11]:
out["figures"]["radius_sensitivity"].show()
out["radius"][["radius", "group", "n_plays", "censored_pct", "median_time_to_pressure_s", "S(2.5s)"]]

,radius,group,n_plays,censored_pct,median_time_to_pressure_s,S(2.5s)
0,1.5,≤3,410,88.8,NaN,0.984
1,1.5,4,5620,81.2,4.8,0.946
2,1.5,5,1618,76.6,3.8,0.889
3,1.5,6+,409,78.0,3.7,0.864
4,2.0,≤3,410,79.0,5.4,0.925
5,2.0,4,5620,67.4,3.6,0.856
6,2.0,5,1618,62.4,3.1,0.775
7,2.0,6+,409,62.1,3.0,0.678
8,3.0,≤3,410,51.2,3.1,0.710
9,3.0,4,5620,37.3,2.7,0.574


## 6. Validation against PFF charting

In [12]:
out["figures"]["validation_boxplots"].show()
out["play_validation"]

,label,score,AUC,point_biserial_r,p_value
0,pressure_pff,PIS (low = pressure),0.7975,-0.5267,0.0
1,pressure_pff,PIS_adj (time-to-throw adjusted),0.7396,-0.4051,0.0
2,pressure_pff,time_to_pressure (censored = clean),0.8029,0.6270,0.0
3,pressure_pff,time_to_throw (baseline),0.7405,0.4190,NaN
4,sack,PIS (low = pressure),0.9509,-0.4654,0.0
5,sack,PIS_adj (time-to-throw adjusted),0.8937,-0.3735,0.0
6,sack,time_to_pressure (censored = clean),0.8597,0.3638,0.0
7,sack,time_to_throw (baseline),0.8529,0.3288,NaN
8,beaten,PIS (low = pressure),0.6878,-0.3342,0.0
9,beaten,PIS_adj (time-to-throw adjusted),0.6702,-0.2980,0.0


In [13]:
out["confusion"]

pressure_pff,PFF: clean,PFF: pressure,All
pressure_event,,,
tracking: pocket held,4569,807,5376
tracking: rusher ≤2 yd,561,2120,2681
All,5130,2927,8057


## 7. Team ranking

In [14]:
out["figures"]["team_ranking"].show()
out["team_validation"]

,team_score,vs,spearman_rho,p_value
0,PIS,pressure_rate_pff,-0.2309,0.2035
1,PIS,sack_rate,-0.0398,0.8289
2,PIS,beaten_rate,-0.4523,0.0093
3,PIS_adj,pressure_rate_pff,-0.1327,0.4691
4,PIS_adj,sack_rate,-0.0207,0.9104
5,PIS_adj,beaten_rate,-0.4380,0.0122


In [15]:
out["teams"][["possessionTeam", "n_plays", "PIS", "PIS_adj", "rank_PIS_adj", "PIS_adj_noPA_noPR", "time_to_throw",
              "pressure_rate_pff", "sack_rate", "beaten_rate", "median_time_to_pressure_s"]].round(3)

,possessionTeam,n_plays,PIS,PIS_adj,rank_PIS_adj,PIS_adj_noPA_noPR,time_to_throw,pressure_rate_pff,sack_rate,beaten_rate,median_time_to_pressure_s
0,ARI,218,66.590,65.048,1,62.928,3.252,0.353,0.087,0.394,4.5
1,BAL,241,63.522,62.650,2,59.032,3.330,0.378,0.087,0.423,3.6
2,KC,307,62.400,60.010,3,60.594,3.152,0.365,0.049,0.365,3.8
3,LA,244,63.130,58.983,4,61.292,2.947,0.283,0.033,0.402,4.4
4,BUF,234,60.694,58.509,5,61.211,3.176,0.423,0.030,0.462,3.6
5,TB,304,60.821,55.115,6,58.941,2.764,0.263,0.040,0.342,4.7
6,PIT,237,61.761,54.970,7,55.350,2.637,0.274,0.055,0.456,3.0
7,CLE,225,57.692,54.914,8,54.667,3.107,0.324,0.093,0.453,3.9
8,NO,181,56.693,54.661,9,57.344,3.194,0.442,0.055,0.470,3.6
9,PHI,221,53.520,53.652,10,52.796,3.448,0.434,0.059,0.425,3.4


Where PIS adds information beyond sacks (top third vs bottom third disagreements):

In [16]:
out["disagree"]

,possessionTeam,profile,sack_rate,PIS_adj,pressure_rate_pff,time_to_throw
0,ARI,many sacks but solid pocket (sacks not on the ...,0.087,65.048,0.353,3.252
1,BAL,many sacks but solid pocket (sacks not on the ...,0.087,62.650,0.378,3.330
7,CLE,many sacks but solid pocket (sacks not on the ...,0.093,54.914,0.324,3.107
22,LAC,few sacks but poor pocket (sacks hide pressure),0.049,46.641,0.299,2.954
24,DAL,few sacks but poor pocket (sacks hide pressure),0.048,45.894,0.314,2.958
28,ATL,few sacks but poor pocket (sacks hide pressure),0.042,41.818,0.368,2.825
31,MIN,few sacks but poor pocket (sacks hide pressure),0.036,36.608,0.376,2.934


Stretch: per-blocker view (share of blocks where the assigned defender got within 2 yd of the QB before the throw, ≥100 blocks).

In [17]:
b = out["blockers"]
pd.concat([b.head(8).assign(group="best"), b.tail(8).assign(group="worst")])

,nflId,team,position,n_blocks,defender_reached_QB_pct,S(2.5s),displayName,officialPosition,group
0,41322,JAX,C,146,2.7,0.986,Brandon Linder,G,best
1,40151,TB,C,287,3.8,0.975,Ryan Jensen,C,best
2,44068,GB,C,127,3.9,0.988,Lucas Patrick,G,best
3,53491,GB,C,102,3.9,1.000,Josh Myers,C,best
4,48159,CHI,C,215,4.2,0.993,Sam Mustipher,C,best
5,46089,DET,C,117,4.3,1.000,Frank Ragnow,C,best
6,41532,NO,LT,104,4.8,1.000,James Hurst,T,best
7,46180,LA,C,228,4.8,1.000,Brian Allen,C,best
166,52426,MIA,LG,263,15.6,0.938,Austin Jackson,T,worst
167,47805,PHI,LT,140,15.7,0.953,Andre Dillard,T,worst


## 8. Animated replays
`viz.replay(gameId, playId)` works for any play. The two examples are picked automatically: the sack with the lowest PIS, and the completed 3-4 s dropback with the highest PIS.

In [18]:
sack, clean = pick_examples(df)
print("sack:", sack, " clean:", clean)
out["figures"]["replay_sack"].show()

sack: (2021102500, 3424)  clean: (2021101707, 306)


In [19]:
out["figures"]["replay_clean"].show()